# VisionBridge A-Z base model training

Run this notebook from top to bottom. It downloads the RealSign ISL alphabet dataset, converts images into VisionBridge's 126D landmark representation, builds a stratified 80/20 train-validation split from the dataset's training + validation pools, and trains automatically.

Colab already provides the PyTorch training runtime. This notebook installs only the MediaPipe dependency needed for landmark extraction, so it does not downgrade Colab's preinstalled FastAPI, Pydantic, httpx, torchvision, or other packages.

The final testing split from the dataset is kept untouched. Training can run for up to 500 epochs and stops early when every A-Z class reaches the configured validation accuracy target. The best checkpoint is saved even when the target is not reached.

The default target is 100% validation accuracy for every letter. That is a validation stopping rule, not a guarantee of perfect live recognition on unseen signers.


In [ ]:
%cd /content
!rm -rf /content/VisionBridge
!git clone --depth 1 https://github.com/BharathWaj-K-R/VisionBridge.git /content/VisionBridge
%cd /content/VisionBridge

# Install only what dataset preparation actually needs.
%pip -q install "mediapipe==0.10.35"

import sys
import torch
import numpy as np
import mediapipe as mp
import cv2

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("MediaPipe:", mp.__version__)
print("OpenCV:", cv2.__version__)

assert mp.__version__ == "0.10.35"
  

In [ ]:
print("MediaPipe dependency: PASS")
print("Colab training environment: READY")


In [ ]:
!rm -rf /content/RealSign /content/RealSign.zip /content/visionbridge_letter_data
!wget -q https://github.com/RealSign62/RealSign-Indian-Sign-Language-Dataset/raw/refs/heads/main/Dataset.zip -O /content/RealSign.zip
!unzip -q /content/RealSign.zip -d /content/RealSign
!python backend/scripts/prepare_letter_dataset.py --input-root /content/RealSign --output-dir /content/visionbridge_letter_data --validation-ratio 0.20 --seed 42


In [ ]:
from pathlib import Path
import json
import numpy as np

root = Path('/content/visionbridge_letter_data')
metadata = json.loads((root / 'labels.json').read_text(encoding='utf-8'))
print('Labels:', ''.join(metadata['labels']))
print('Split policy:', metadata['split_policy'])
for split in ('train', 'val', 'test'):
    data = np.load(root / f'{split}.npz')
    print(f'{split}: samples={len(data["x"])} features={data["x"].shape}')


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '/content/VisionBridge/backend')
from app.training.letter_base import train_model

DATA_DIR = Path('/content/visionbridge_letter_data')
OUTPUT = Path('/content/VisionBridge/backend/app/models/weights/letter_base_model.pt')

result = train_model(
    root=DATA_DIR,
    output_path=OUTPUT,
    epochs=500,
    batch_size=128,
    learning_rate=1e-3,
    weight_decay=1e-4,
    target_class_accuracy=1.0,
    seed=42,
    hidden_dim=128,
    embedding_dim=64,
    dropout=0.10,
)

print('\nTraining result:')
print('target_reached =', result['reached_target'])
print('test_accuracy =', f"{result['test_accuracy']:.4f}")


In [ ]:
import sys
sys.path.insert(0, '/content/VisionBridge/backend')
from app.models.letter_model import load_checkpoint

checkpoint = '/content/VisionBridge/backend/app/models/weights/letter_base_model.pt'
model = load_checkpoint(checkpoint)
print('CHECKPOINT LOAD: PASS')
print('input_dim =', model.input_dim)
print('hidden_dim =', model.hidden_dim)
print('embedding_dim =', model.embedding_dim)
print('classes =', model.num_classes)
print('labels =', ''.join(model.labels))


## After the run

The trained checkpoint is saved at `backend/app/models/weights/letter_base_model.pt`. Download that file into your local VisionBridge repository at the same path.

The stopping condition is based on the validation split. The final test accuracy is reported separately so the test set remains an honest held-out measurement.
